# Inspect activation precision in `voc_8n_best_int8.onnx`

这个 notebook 检查 ONNX 图中的量化形式，并回答以下问题：

1. 模型是否包含 INT8 权重和量化节点？
2. YOLOv8 的 `Sigmoid + Mul` 是否构成 SiLU？
3. SiLU 节点本身接收和输出的是 INT8 还是 FLOAT？

> 注意：ONNX 图能说明模型的逻辑数据类型和 Q/DQ 边界，但最终硬件是否融合这些节点并使用 INT8 kernel，需要结合具体执行后端的 profiling 才能完全确认。

In [1]:
from pathlib import Path
from collections import Counter, defaultdict

import onnx
import pandas as pd
from IPython.display import display, Markdown
from onnx import TensorProto, shape_inference

MODEL_RELATIVE_PATH = Path("original/yolov8n_voc/voc_8n_best_int8.onnx")

def locate_model(relative_path: Path) -> Path:
    start = Path.cwd().resolve()
    for base in (start, *start.parents):
        candidate = base / relative_path
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"找不到 {relative_path}。请从 Dissertation 项目或 Activation 目录运行此 notebook。"
    )

model_path = locate_model(MODEL_RELATIVE_PATH)
model = onnx.load(model_path, load_external_data=False)
onnx.checker.check_model(model)
inferred_model = shape_inference.infer_shapes(model)

print(f"Model: {model_path}")
print(f"Size: {model_path.stat().st_size / 1024**2:.2f} MiB")
print(f"IR version: {model.ir_version}")
print(f"Opset: {[(item.domain or 'ai.onnx', item.version) for item in model.opset_import]}")
print("ONNX checker: passed")

Model: /Users/richard/Developer/Dissertation/original/yolov8n_voc/voc_8n_best_int8.onnx
Size: 3.24 MiB
IR version: 8
Opset: [('ai.onnx', 18)]
ONNX checker: passed


## 1. 模型输入、输出与算子统计

Q/DQ 模型通常使用 `QuantizeLinear` 和 `DequantizeLinear` 标记量化边界。`QLinearConv` 或 `ConvInteger` 则是另一种显式整数算子表示。

In [2]:
def value_info_row(value):
    tensor_type = value.type.tensor_type
    shape = [dim.dim_value if dim.dim_value else dim.dim_param for dim in tensor_type.shape.dim]
    return {
        "name": value.name,
        "dtype": TensorProto.DataType.Name(tensor_type.elem_type),
        "shape": shape,
    }

display(Markdown("**Model inputs**"))
display(pd.DataFrame(value_info_row(value) for value in model.graph.input))
display(Markdown("**Model outputs**"))
display(pd.DataFrame(value_info_row(value) for value in model.graph.output))

operator_counts = Counter(node.op_type for node in model.graph.node)
operator_table = pd.DataFrame(
    operator_counts.most_common(), columns=["operator", "count"]
)
display(Markdown("**Operator counts**"))
display(operator_table)

**Model inputs**

,name,dtype,shape
0,images,FLOAT,"[1, 3, 640, 640]"


**Model outputs**

,name,dtype,shape
0,output0,FLOAT,"[1, 24, 8400]"


**Operator counts**

,operator,count
0,DequantizeLinear,250
1,QuantizeLinear,123
2,Conv,64
3,Sigmoid,57
4,Mul,57
5,Concat,17
6,Split,8
7,Add,8
8,Reshape,8
9,MaxPool,3


In [3]:
quantization_operators = [
    "QuantizeLinear",
    "DequantizeLinear",
    "QLinearConv",
    "ConvInteger",
    "MatMulInteger",
    "DynamicQuantizeLinear",
]

quantization_table = pd.DataFrame(
    {"operator": op, "count": operator_counts[op]}
    for op in quantization_operators
)
initializer_types = Counter(
    TensorProto.DataType.Name(item.data_type) for item in model.graph.initializer
)
initializer_table = pd.DataFrame(
    initializer_types.items(), columns=["initializer dtype", "count"]
).sort_values("count", ascending=False)

display(Markdown("**Quantization operators**"))
display(quantization_table)
display(Markdown("**Initializer data types**"))
display(initializer_table)

is_qdq_model = (
    operator_counts["QuantizeLinear"] > 0
    and operator_counts["DequantizeLinear"] > 0
)
has_int8_parameters = initializer_types["INT8"] > 0
print(f"Q/DQ graph: {is_qdq_model}")
print(f"Contains INT8 initializers: {has_int8_parameters}")

**Quantization operators**

,operator,count
0,QuantizeLinear,123
1,DequantizeLinear,250
2,QLinearConv,0
3,ConvInteger,0
4,MatMulInteger,0
5,DynamicQuantizeLinear,0


**Initializer data types**

,initializer dtype,count
1,FLOAT,255
2,INT8,251
3,INT32,126
0,INT64,12


Q/DQ graph: True
Contains INT8 initializers: True


## 2. 找出 `Sigmoid + Mul` 形式的 SiLU

SiLU 定义为 `SiLU(x) = x * sigmoid(x)`。如果一个 `Sigmoid` 的输出进入 `Mul`，而 `Mul` 的另一个输入正是同一个 `x`，这里就把它识别为 SiLU。检测头中单独使用、没有这种乘法结构的 `Sigmoid` 不属于 SiLU。

In [4]:
producer_by_tensor = {
    output: node for node in model.graph.node for output in node.output
}
consumers_by_tensor = defaultdict(list)
for node in model.graph.node:
    for input_name in node.input:
        consumers_by_tensor[input_name].append(node)

tensor_dtype = {}
for value in (
    list(inferred_model.graph.input)
    + list(inferred_model.graph.value_info)
    + list(inferred_model.graph.output)
):
    if value.type.HasField("tensor_type"):
        elem_type = value.type.tensor_type.elem_type
        tensor_dtype[value.name] = TensorProto.DataType.Name(elem_type)

silu_rows = []
standalone_sigmoid_rows = []

for sigmoid in (node for node in model.graph.node if node.op_type == "Sigmoid"):
    x_name = sigmoid.input[0]
    sigmoid_output = sigmoid.output[0]
    matching_mul = next(
        (
            node
            for node in consumers_by_tensor[sigmoid_output]
            if node.op_type == "Mul" and x_name in node.input
        ),
        None,
    )

    if matching_mul is None:
        standalone_sigmoid_rows.append(
            {
                "sigmoid": sigmoid.name,
                "input dtype": tensor_dtype.get(x_name, "unknown"),
                "output dtype": tensor_dtype.get(sigmoid_output, "unknown"),
                "consumer ops": ", ".join(
                    node.op_type for node in consumers_by_tensor[sigmoid_output]
                ),
            }
        )
        continue

    silu_output = matching_mul.output[0]
    input_producer = producer_by_tensor.get(x_name)
    output_consumers = consumers_by_tensor[silu_output]
    silu_rows.append(
        {
            "SiLU": sigmoid.name.removesuffix("/Sigmoid"),
            "input producer": input_producer.op_type if input_producer else "graph input",
            "input dtype": tensor_dtype.get(x_name, "unknown"),
            "sigmoid dtype": tensor_dtype.get(sigmoid_output, "unknown"),
            "mul output dtype": tensor_dtype.get(silu_output, "unknown"),
            "next ops": ", ".join(node.op_type for node in output_consumers),
            "immediate requantize": any(
                node.op_type == "QuantizeLinear" for node in output_consumers
            ),
        }
    )

silu_table = pd.DataFrame(silu_rows)
standalone_sigmoid_table = pd.DataFrame(standalone_sigmoid_rows)

print(f"Detected SiLU patterns: {len(silu_table)}")
print(f"Standalone Sigmoid nodes: {len(standalone_sigmoid_table)}")
display(silu_table)
display(Markdown("**Standalone Sigmoid nodes (for example, detection-head probabilities)**"))
display(standalone_sigmoid_table)

Detected SiLU patterns: 56
Standalone Sigmoid nodes: 1


,SiLU,input producer,input dtype,sigmoid dtype,mul output dtype,next ops,immediate requantize
0,/model.0/act,DequantizeLinear,FLOAT,FLOAT,FLOAT,QuantizeLinear,True
1,/model.1/act,DequantizeLinear,FLOAT,FLOAT,FLOAT,QuantizeLinear,True
2,/model.2/cv1/act,DequantizeLinear,FLOAT,FLOAT,FLOAT,Split,False
3,/model.2/m.0/cv1/act,DequantizeLinear,FLOAT,FLOAT,FLOAT,QuantizeLinear,True
4,/model.2/m.0/cv2/act,DequantizeLinear,FLOAT,FLOAT,FLOAT,Add,False
5,/model.2/cv2/act,DequantizeLinear,FLOAT,FLOAT,FLOAT,QuantizeLinear,True
6,/model.3/act,DequantizeLinear,FLOAT,FLOAT,FLOAT,QuantizeLinear,True
7,/model.4/cv1/act,DequantizeLinear,FLOAT,FLOAT,FLOAT,Split,False
8,/model.4/m.0/cv1/act,DequantizeLinear,FLOAT,FLOAT,FLOAT,QuantizeLinear,True
9,/model.4/m.0/cv2/act,DequantizeLinear,FLOAT,FLOAT,FLOAT,Add,False


**Standalone Sigmoid nodes (for example, detection-head probabilities)**

,sigmoid,input dtype,output dtype,consumer ops
0,/model.22/Sigmoid,FLOAT,FLOAT,Concat


## 3. 自动生成结论

In [5]:
all_silu_float = (
    not silu_table.empty
    and silu_table["input dtype"].eq("FLOAT").all()
    and silu_table["sigmoid dtype"].eq("FLOAT").all()
    and silu_table["mul output dtype"].eq("FLOAT").all()
)
all_silu_inputs_dequantized = (
    not silu_table.empty
    and silu_table["input producer"].eq("DequantizeLinear").all()
)
immediate_requantized_count = int(silu_table["immediate requantize"].sum())

if is_qdq_model and has_int8_parameters and all_silu_float:
    conclusion = f"""
### 结论

- 这是一个 **Q/DQ 格式的 INT8 量化模型**：图中有 {operator_counts['QuantizeLinear']} 个 `QuantizeLinear`、{operator_counts['DequantizeLinear']} 个 `DequantizeLinear`，并包含 {initializer_types['INT8']} 个 INT8 initializer。
- 共识别到 **{len(silu_table)} 个 SiLU**（`Sigmoid + Mul`）和 **{len(standalone_sigmoid_table)} 个单独的 Sigmoid**。
- 所有 SiLU 的输入、Sigmoid 输出和 Mul 输出，在 ONNX 类型系统中都是 **FLOAT**。
- 所有 SiLU 输入前面都是 `DequantizeLinear`：**INT8/量化表示先被反量化为 FLOAT，然后执行 SiLU**。
- 有 {immediate_requantized_count} 个 SiLU 输出紧接 `QuantizeLinear`；其余输出先进入 Split、Add、Concat、Resize 等浮点图节点，之后可能在更下游重新量化。

因此，就这个 ONNX 图本身而言，**SiLU activation 不是直接以 INT8 执行的**。模型的权重和许多张量边界经过 INT8 量化，但 SiLU 被表示为 FLOAT `Sigmoid + Mul`。

如果部署后端支持图融合，它可能消除部分 Q/DQ 或选择专用 kernel；是否在硬件上真正执行 INT8 仍应通过 TensorRT/ONNX Runtime/OpenVINO 的执行计划或 profiler 验证。
"""
else:
    conclusion = "自动规则无法得到单一结论，请检查上面的逐层表格。"

display(Markdown(conclusion))


### 结论

- 这是一个 **Q/DQ 格式的 INT8 量化模型**：图中有 123 个 `QuantizeLinear`、250 个 `DequantizeLinear`，并包含 251 个 INT8 initializer。
- 共识别到 **56 个 SiLU**（`Sigmoid + Mul`）和 **1 个单独的 Sigmoid**。
- 所有 SiLU 的输入、Sigmoid 输出和 Mul 输出，在 ONNX 类型系统中都是 **FLOAT**。
- 所有 SiLU 输入前面都是 `DequantizeLinear`：**INT8/量化表示先被反量化为 FLOAT，然后执行 SiLU**。
- 有 34 个 SiLU 输出紧接 `QuantizeLinear`；其余输出先进入 Split、Add、Concat、Resize 等浮点图节点，之后可能在更下游重新量化。

因此，就这个 ONNX 图本身而言，**SiLU activation 不是直接以 INT8 执行的**。模型的权重和许多张量边界经过 INT8 量化，但 SiLU 被表示为 FLOAT `Sigmoid + Mul`。

如果部署后端支持图融合，它可能消除部分 Q/DQ 或选择专用 kernel；是否在硬件上真正执行 INT8 仍应通过 TensorRT/ONNX Runtime/OpenVINO 的执行计划或 profiler 验证。


## 4. 对直接 INT8 LUT 的意义

当前图中的典型路径是：

```text
量化张量 → DequantizeLinear → FLOAT Sigmoid → FLOAT Mul → QuantizeLinear
```

理论上可以研究把完整路径替换为：

```text
INT8 输入 → 256-entry INT8 LUT → INT8 输出
```

但 LUT 必须针对每一处 activation 的输入/输出 `scale` 和 `zero_point` 生成；如果各层量化参数不同，就不能让所有 SiLU 共用同一张表。ONNX 标准图也没有一个通用的 256 项 INT8 SiLU-LUT 算子，通常需要图重写、自定义算子或目标推理后端的插件支持。